# RAG Systems: Retrieval-Augmented Generation

RAG is the most common LLM system design pattern in production. This note implements BM25 retrieval, hybrid search, chunking strategies, and evaluates recall@k and MRR — the metrics interviewers expect you to know.

## What Interviewers Test
- Chunking strategies and their tradeoffs
- Dense vs sparse (BM25) vs hybrid retrieval — when each dominates
- Re-ranking with a cross-encoder
- Evaluation: recall@k, MRR, NDCG@k
- Failure modes: context overflow, irrelevant retrieval, lost-in-the-middle
- When to fine-tune embeddings vs use off-the-shelf

In [ ]:
import numpy as np
from collections import Counter
import math
np.random.seed(42)

# ===== BM25 Retrieval =====
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs, self.tokenized, self.idf, self.avg_dl = [], [], {}, 0

    def fit(self, documents):
        self.docs = documents
        self.tokenized = [d.lower().split() for d in documents]
        N = len(documents)
        self.avg_dl = np.mean([len(t) for t in self.tokenized])
        df = Counter(tok for doc in self.tokenized for tok in set(doc))
        self.idf = {t: math.log((N - df[t] + 0.5) / (df[t] + 0.5) + 1) for t in df}
        return self

    def score(self, query, doc_idx):
        query_terms = query.lower().split()
        doc = self.tokenized[doc_idx]
        dl = len(doc)
        tf = Counter(doc)
        score = 0.0
        for term in query_terms:
            if term in tf:
                f = tf[term]
                score += self.idf.get(term, 0) * (
                    f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self.avg_dl))
                )
        return score

    def retrieve(self, query, top_k=5):
        scores = [self.score(query, i) for i in range(len(self.docs))]
        top_idx = np.argsort(scores)[::-1][:top_k]
        return top_idx, [scores[i] for i in top_idx]

# Sample document corpus
documents = [
    "Machine learning models require large amounts of training data to perform well.",
    "Transformer architectures use self-attention mechanisms for sequence modeling.",
    "RAG combines retrieval systems with language model generation for grounded answers.",
    "Gradient descent optimizes neural network weights by minimizing a loss function.",
    "BERT is a bidirectional encoder pre-trained on masked language modeling tasks.",
    "Large language models can perform few-shot learning with in-context examples.",
    "Retrieval augmented generation improves factual accuracy in language model outputs.",
    "Attention mechanisms allow models to focus on relevant parts of the input sequence.",
    "Fine-tuning adapts pretrained models to specific downstream tasks efficiently.",
    "Vector databases store embeddings for fast approximate nearest neighbor search.",
]

bm25 = BM25().fit(documents)
query = "how does retrieval help language models"
idx, scores = bm25.retrieve(query, top_k=5)
print("BM25 Top-5 results:")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  {rank}. [{s:.3f}] {documents[i][:70]}")


In [ ]:
# ===== Dense Retrieval (simulated) + Hybrid Fusion =====
def random_embedding(text, dim=32, seed=None):
    """Simulate a text embedding (random for demo; in prod: sentence-transformers)."""
    if seed is None:
        seed = abs(hash(text)) % (2**31)
    rng = np.random.RandomState(seed)
    emb = rng.randn(dim)
    return emb / (np.linalg.norm(emb) + 1e-8)

doc_embs = np.array([random_embedding(d) for d in documents])

def dense_retrieve(query, doc_embs, top_k=5):
    q_emb = random_embedding(query)
    scores = doc_embs @ q_emb
    idx = np.argsort(-scores)[:top_k]
    return idx, scores[idx]

def reciprocal_rank_fusion(rankings_list, k=60):
    """
    RRF: fuse multiple ranked lists without score normalization.
    score(doc) = sum over rankings: 1 / (k + rank_of_doc)
    k=60 is the standard default.
    """
    doc_scores = Counter()
    for ranking in rankings_list:
        for rank, doc_idx in enumerate(ranking, 1):
            doc_scores[doc_idx] += 1.0 / (k + rank)
    return sorted(doc_scores.keys(), key=lambda d: -doc_scores[d])

# Hybrid retrieval
bm25_idx, _ = bm25.retrieve(query, top_k=10)
dense_idx, _ = dense_retrieve(query, doc_embs, top_k=10)
hybrid_idx = reciprocal_rank_fusion([list(bm25_idx), list(dense_idx)])

print("Retrieval comparison for query:", repr(query))
print(f"{'Rank':<6} {'BM25':>5} {'Dense':>6} {'Hybrid':>7}")
for i in range(5):
    b = bm25_idx[i] if i < len(bm25_idx) else '-'
    d = dense_idx[i] if i < len(dense_idx) else '-'
    h = hybrid_idx[i] if i < len(hybrid_idx) else '-'
    print(f"  {i+1}:  doc{b:<4} doc{d:<5} doc{h}")


## Chunking Strategies Comparison

| Strategy | How | Good for | Risk |
|---|---|---|---|
| **Fixed-size** | Split every N tokens with M-token overlap | Uniform docs | Splits mid-sentence |
| **Sentence** | Split on `.`, `!`, `?` | Narrative/FAQ text | Variable chunk size |
| **Recursive** | Split on `\n\n`, `\n`, ` ` in priority order | General purpose | May miss semantic splits |
| **Semantic** | Split where embedding similarity drops | Mixed-topic docs | Slow, needs embedding |
| **Document structure** | Split on headers/sections | Markdown, HTML | Requires parsing |

**Chunk size rule of thumb:** 200–500 tokens. Too small = loss of context; too large = retrieval imprecision.


In [ ]:
# ===== Retrieval Evaluation: Recall@k and MRR =====
def recall_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of relevant docs found in top-k retrieved."""
    top_k = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    return len(top_k & relevant) / (len(relevant) + 1e-8)

def mrr(retrieved_ids, relevant_ids):
    """Mean Reciprocal Rank: 1/rank of first relevant doc."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved_ids, relevance_grades, k):
    """NDCG@k: graded relevance, discounted by position."""
    def dcg(ids, grades, k):
        return sum(grades.get(doc, 0) / math.log2(rank + 1)
                   for rank, doc in enumerate(ids[:k], 1))
    ideal_order = sorted(relevance_grades.keys(), key=lambda d: -relevance_grades[d])
    idcg = dcg(ideal_order, relevance_grades, k)
    return dcg(retrieved_ids, relevance_grades, k) / (idcg + 1e-8)

# Simulate evaluation
queries = {
    'Q1': {'relevant': [2, 6], 'retrieved_bm25': [6, 2, 0, 3, 7], 'retrieved_dense': [2, 5, 6, 1, 8]},
    'Q2': {'relevant': [1, 7], 'retrieved_bm25': [0, 3, 7, 1, 5], 'retrieved_dense': [7, 1, 2, 4, 3]},
    'Q3': {'relevant': [3, 8], 'retrieved_bm25': [3, 1, 2, 8, 5], 'retrieved_dense': [8, 0, 3, 6, 2]},
}

for method in ['bm25', 'dense']:
    rec1 = np.mean([recall_at_k(q[f'retrieved_{method}'], q['relevant'], 1) for q in queries.values()])
    rec3 = np.mean([recall_at_k(q[f'retrieved_{method}'], q['relevant'], 3) for q in queries.values()])
    mrr_  = np.mean([mrr(q[f'retrieved_{method}'], q['relevant']) for q in queries.values()])
    print(f"{method:>8}: Recall@1={rec1:.3f}, Recall@3={rec3:.3f}, MRR={mrr_:.3f}")


## Common Interview Questions

**Q: When does BM25 outperform dense retrieval, and vice versa?**
BM25 wins on keyword-intensive queries where exact term matching matters ("AAPL Q3 earnings", "Python AttributeError"). Dense retrieval wins on semantic queries where paraphrase matters ("how do I fix a type error" vs "AttributeError"). Hybrid retrieval (RRF fusion) consistently outperforms either alone by covering both cases.

**Q: What is the "lost-in-the-middle" problem?**
LLMs tend to pay more attention to content at the beginning and end of the context window, and relatively less to information in the middle. When long RAG contexts are assembled, relevant chunks placed in the middle may be underutilized. Mitigation: order retrieved chunks by relevance (most relevant first and last), use re-ranking, and keep contexts short.

**Q: How do you evaluate a RAG system?**
Component-level: retrieval recall@k and MRR (does the retriever find the right docs?). End-to-end: answer correctness on a golden set (exact match for factual queries, LLM-as-judge for open-ended). Online: user thumbs, resolution rate, follow-up question rate. A RAG system can fail at retrieval, at generation, or at the grounding step — evaluate each separately.

**Q: When would you fine-tune the embedding model vs use off-the-shelf?**
Off-the-shelf (e.g., BGE, E5, OpenAI embeddings) works well for general knowledge. Fine-tune when: domain vocabulary is very specialized (medical, legal), relevance signals differ from general web (code search, product search), or retrieval evaluation shows persistent failures on domain queries. Fine-tune on (query, positive_passage, negative_passage) triplets with contrastive loss.

## Key Takeaways
- RAG = retrieval (find relevant chunks) + augmentation (inject into context) + generation (produce answer)
- BM25: keyword match, O(N) retrieval; dense: semantic, O(log N) with ANN; hybrid RRF: consistently best
- Chunking: 200–500 tokens with overlap; strategy depends on document structure
- Evaluation: recall@k / MRR for retrieval component; LLM-as-judge or golden set for end-to-end
- Lost-in-the-middle: order context so relevant chunks are at start/end; keep contexts short
- Fine-tune embeddings when off-the-shelf recall@k on your domain is < 70%